# **Baseline : Regresja logistyczna**
## Analiza ryzyka płynności przedsiębiorstw
&emsp;Celem jest stworzenie punktu odniesienia (baseline) przewidującego prawdopodobieństwo opóźnienia płatności dla zamkniętych faktur.


### Narzędzia i biblioteki
&emsp; Zacznijmy od załadowania niezbędnych narędzi.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (11.7, 7)

### Wczytanie i filtracja danych oraz selekcja cech
&emsp;W tym kroku przygotujemy macierz danych do treningu modelu. Po wczytaniu zbioru odrzucimy wszystkie faktury otwarte (ocenzurowane), ponieważ klasyfikator binarny wymaga danych wyłącznie z zamkniętych transakcji o jednoznacznym statusie (zapłacone w terminie lub opóźnione). Następnie przeprowadzimy selekcję cech usuwając identyfikatory tekstowe, daty oraz kolumny zawierające dane o przyszłości z perspektywy momentu wystawienia faktury: np. kolumnę *days_late*. Model powinien uczyć się wyłącznie na danych, które są dostępne w momencie, w którym ma być wykorzystany do predykcji.

In [28]:
df_input = pd.read_csv('../data/dataset_survival.csv')
print(f"Wymiary macierzy wejściowej: {df_input.shape[0]} wierszy, {df_input.shape[1]} kolumn")

df_closed = df_input[df_input['isOpen'] == 0].copy()

leakage_cols = ['clear_date', 'days_late', 'event', 'time_days', 'isOpen']

id_cols = ['doc_id', 'invoice_id', 'cust_number', 'name_customer']
date_cols = ['document_create_date', 'due_in_date', 'baseline_create_date', 'buisness_year']

cols_to_drop = leakage_cols + id_cols + date_cols

df_closed.drop(columns=cols_to_drop, axis=1, inplace=True)

cols = df_closed.columns.tolist()
cols.remove("paid_late")
cols.append("paid_late")
df_closed = df_closed[cols]

print(f"Wymiary macierzy po obróbce: {df_closed.shape[0]} wierszy, {df_closed.shape[1]} kolumn")
print("\nWektor wejściowy (Features + Target):")
print(cols)

Wymiary macierzy wejściowej: 49999 wierszy, 19 kolumn
Wymiary macierzy po obróbce: 39999 wierszy, 6 kolumn

Wektor wejściowy (Features + Target):
['business_code', 'invoice_currency', 'total_open_amount', 'cust_payment_terms', 'segment', 'paid_late']
